# SPY Options Chain Analysis - 04/02/2026 Expiry

Analysis of SPY options chain data including IV smile, Greeks, and SD bands.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('spy_options_clean.csv')
df_calls = df[df['Type'] == 'Call'].sort_values('Strike').reset_index(drop=True)
df_puts  = df[df['Type'] == 'Put'].sort_values('Strike').reset_index(drop=True)
S = 560.0   # approximate SPY spot price on 3/27/2026
r = 0.045   # risk-free rate
T = 6/365   # days to expiry: 04/02/2026
print(df.head())

In [ ]:
def d1(S, K, T, r, sigma):
    return (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def calc_delta(S, K, T, r, sigma, opt_type):
    val = d1(S, K, T, r, sigma)
    if opt_type == 'Call':
        return norm.cdf(val)
    return norm.cdf(val) - 1

def calc_gamma(S, K, T, r, sigma):
    return norm.pdf(d1(S, K, T, r, sigma)) / (S * sigma * np.sqrt(T))

df_calls['Delta'] = df_calls.apply(lambda row: calc_delta(S, row['Strike'], T, r, row['IV'], 'Call'), axis=1)
df_calls['Gamma'] = df_calls.apply(lambda row: calc_gamma(S, row['Strike'], T, r, row['IV']), axis=1)

atm_iv = df_calls.iloc[(df_calls['Strike'] - S).abs().argsort()[:1]]['IV'].values[0]
move_1sd = S * atm_iv * np.sqrt(T)
sd_levels = {
    '1 SD':   (S - move_1sd,       S + move_1sd),
    '2 SD':   (S - 2 * move_1sd,   S + 2 * move_1sd),
    '2.5 SD': (S - 2.5 * move_1sd, S + 2.5 * move_1sd)
}
print(f'ATM IV: {atm_iv:.2%}, 1SD move: +/-{move_1sd:.2f}')

In [ ]:
def plot_sd_bands(ax):
    colors = {'1 SD': 'green', '2 SD': 'orange', '2.5 SD': 'red'}
    for label, (lower, upper) in sd_levels.items():
        ax.axvline(lower, color=colors[label], linestyle='--', alpha=0.6, linewidth=1)
        ax.axvline(upper, color=colors[label], linestyle='--', alpha=0.6, linewidth=1, label=f'{label}: [{lower:.1f}, {upper:.1f}]')
    ax.axvline(S, color='black', linestyle='-', linewidth=1.5, label=f'Spot ({S})')

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 15), sharex=True)

ax1.plot(df_calls['Strike'], df_calls['IV'] * 100, marker='o', color='steelblue', linewidth=2, label='Call IV')
ax1.plot(df_puts['Strike'],  df_puts['IV']  * 100, marker='s', color='salmon',    linewidth=2, label='Put IV')
ax1.set_title('SPY Implied Volatility Smile - 04/02/2026 Expiry', fontsize=13, fontweight='bold')
ax1.set_ylabel('Implied Volatility (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)
plot_sd_bands(ax1)
ax1.legend()

ax2.plot(df_calls['Strike'], df_calls['Delta'], marker='o', color='purple', linewidth=2)
ax2.set_title('Call Delta by Strike', fontsize=13, fontweight='bold')
ax2.set_ylabel('Delta')
ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='ATM (delta=0.5)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plot_sd_bands(ax2)

ax3.plot(df_calls['Strike'], df_calls['Gamma'], marker='o', color='crimson', linewidth=2)
ax3.set_title('Gamma Risk Profile', fontsize=13, fontweight='bold')
ax3.set_xlabel('Strike Price')
ax3.set_ylabel('Gamma')
ax3.grid(True, alpha=0.3)
plot_sd_bands(ax3)

fig.suptitle('SPY Options Analysis - 04/02/2026', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('spy_options_analysis.png', dpi=150, bbox_inches='tight')
print('Saved spy_options_analysis.png')

In [ ]:
print('=== CALL SUMMARY ===')
print(df_calls[['Strike', 'Last', 'Bid', 'Ask', 'IV', 'Delta', 'Gamma']].to_string(index=False))
print()
print('=== PUT SUMMARY ===')
print(df_puts[['Strike', 'Last', 'Bid', 'Ask', 'IV']].to_string(index=False))
print()
print(f'ATM IV: {atm_iv:.4f} ({atm_iv:.2%})')
print(f'Expected 1SD move (6d): +/- {move_1sd:.2f} pts')
print(f'1SD range: [{S - move_1sd:.2f}, {S + move_1sd:.2f}]')
print(f'2SD range: [{S - 2*move_1sd:.2f}, {S + 2*move_1sd:.2f}]')